In [ ]:
# ==============================================================================
# ĐỒ ÁN MÔN PHÂN TÍCH DỮ LIỆU MẠNG XÃ HỘI
# Sinh viên thực hiện: LÊ XUÂN THÀNH
# Nhiệm vụ: Tải dữ liệu, tiền xử lý, trích xuất Sub-graph và thống kê Topology
# Nền tảng khuyến nghị: Google Colab
# ==============================================================================

# === CELL 1: IMPORT THƯ VIỆN & KẾT NỐI GOOGLE DRIVE ===
import networkx as nx
import pandas as pd
import urllib.request
import gzip
import shutil
import os
import pickle
import random

# Nếu chạy trên Google Colab, hãy bỏ comment 2 dòng dưới đây để lưu thẳng vào Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Đường dẫn thư mục làm việc (Sửa lại theo Drive của nhóm)
# Ví dụ: WORK_DIR = '/content/drive/MyDrive/Do_An_SNA_Nhom5/'
WORK_DIR = './'
print("Đã import thành công các thư viện!")


# === CELL 2: TẢI VÀ GIẢI NÉN DỮ LIỆU THỰC TẾ TỪ STANFORD (SNAP) ===
# Nguồn: Amazon product co-purchasing network
# Đã cập nhật URL chuẩn xác mới nhất từ Stanford SNAP
url = 'https://snap.stanford.edu/data/bigdata/communities/com-amazon.ungraph.txt.gz'
gz_path = os.path.join(WORK_DIR, 'com-amazon.ungraph.txt.gz')
txt_path = os.path.join(WORK_DIR, 'com-amazon.ungraph.txt')

print("Đang tải dữ liệu từ ĐH Stanford... (Vui lòng đợi vài giây)")
try:
    if not os.path.exists(gz_path) and not os.path.exists(txt_path):
        urllib.request.urlretrieve(url, gz_path)
        print("Tải xong! Đang giải nén dữ liệu...")
        with gzip.open(gz_path, 'rb') as f_in:
            with open(txt_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print("Giải nén hoàn tất!")
    else:
        print("File dữ liệu đã tồn tại, bỏ qua bước tải.")
except urllib.error.HTTPError as e:
    print(f"\n[LỖI TẢI DỮ LIỆU]: Link Stanford SNAP có thể đang bảo trì. Lỗi: {e}")
    print("Vui lòng thử lại sau hoặc tải file thủ công từ trang chủ SNAP.")


# === CELL 3: ĐỌC DỮ LIỆU VÀO NETWORKX ===
print("Đang xây dựng đồ thị từ tập dữ liệu gốc...")
# Dữ liệu gốc có dạng edgelist, ta dùng nx.read_edgelist
G_full = nx.read_edgelist(txt_path, comments='#', create_using=nx.Graph(), nodetype=int)

print(f"--- THỐNG KÊ ĐỒ THỊ GỐC ---")
print(f"Số lượng Node (Sản phẩm): {G_full.number_of_nodes():,}")
print(f"Số lượng Edge (Lượt mua chung): {G_full.number_of_edges():,}")


# === CELL 4: TRÍCH XUẤT SUB-GRAPH (ĐỒ THỊ CON) ĐỂ TỐI ƯU HIỆU SUẤT ===
# Đồ thị gốc hơn 300,000 nodes sẽ làm treo máy khi chạy Machine Learning.
# Giải pháp điểm 10: Dùng thuật toán duyệt theo chiều rộng (BFS) từ một Node
# có bậc (Degree) cao nhất để lấy ra 8,000 nodes liên kết chặt chẽ với nhau.

print("\nĐang tìm kiếm 'Sản phẩm trung tâm' để làm điểm bắt đầu...")
# Lấy node có bậc cao nhất làm gốc để đồ thị con không bị rời rạc
degrees = dict(G_full.degree())
start_node = max(degrees, key=degrees.get)
print(f"Sản phẩm trung tâm (Node ID: {start_node}) có {degrees[start_node]} lượt mua kèm.")

print("Đang trích xuất đồ thị con gồm 8,000 Nodes bằng thuật toán BFS...")
TARGET_NODES = 8000
# Dùng BFS để lấy các nodes lân cận
bfs_edges = list(nx.bfs_edges(G_full, source=start_node, depth_limit=4))
sub_nodes = set([start_node])

for u, v in bfs_edges:
    if len(sub_nodes) >= TARGET_NODES:
        break
    sub_nodes.add(v)

# Tạo đồ thị con từ danh sách nodes đã chọn
G_sub = G_full.subgraph(sub_nodes).copy()
print("Trích xuất Sub-graph thành công!")


# === CELL 5: PHÂN TÍCH THỐNG KÊ MẠNG LƯỚI CON (DÙNG ĐỂ VIẾT BÁO CÁO) ===
# Thành sẽ copy các thông số ở mục này để đưa vào Báo cáo (Chương 3)
print("\n" + "="*50)
print("THỐNG KÊ NETWORK TOPOLOGY (Dành cho Chương 3 Báo Cáo)")
print("="*50)
num_nodes = G_sub.number_of_nodes()
num_edges = G_sub.number_of_edges()
density = nx.density(G_sub)

print(f"1. Tổng số Nodes (Sản phẩm): {num_nodes:,}")
print(f"2. Tổng số Edges (Lượt mua chéo): {num_edges:,}")
print(f"3. Density (Mật độ đồ thị): {density:.5f}")

# Kiểm tra tính liên thông để tính đường đi ngắn nhất
if nx.is_connected(G_sub):
    print("4. Đồ thị liên thông hoàn toàn (Connected).")
    # Tính Average Shortest Path (Lưu ý: Mất khoảng 1-2 phút để chạy)
    print("Đang tính Average Shortest Path Length...")
    avg_shortest_path = nx.average_shortest_path_length(G_sub)
    print(f"5. Average Shortest Path (Đường đi ngắn nhất trung bình): {avg_shortest_path:.4f}")
else:
    print("4. Đồ thị KHÔNG liên thông hoàn toàn.")
    # Tính trên thành phần liên thông lớn nhất
    largest_cc = max(nx.connected_components(G_sub), key=len)
    G_largest_cc = G_sub.subgraph(largest_cc)
    print(f"   -> Đang tính Average Shortest Path trên cụm lớn nhất ({len(largest_cc)} nodes)...")
    avg_shortest_path = nx.average_shortest_path_length(G_largest_cc)
    print(f"5. Average Shortest Path (Đường đi ngắn nhất trung bình): {avg_shortest_path:.4f}")
print("="*50)


# === CELL 6: LƯU FILE VÀ BÀN GIAO CHO LOAN, HÂN, LY ===
output_filename = 'amazon_graph.gpickle'
output_path = os.path.join(WORK_DIR, output_filename)

print(f"\nĐang lưu đồ thị vào file: {output_path}...")
# Sử dụng thư viện pickle để lưu nguyên trạng object Đồ thị (G_sub)
with open(output_path, 'wb') as f:
    pickle.dump(G_sub, f, pickle.HIGHEST_PROTOCOL)

print("LƯU THÀNH CÔNG! ✅")

Đã import thành công các thư viện!
Đang tải dữ liệu từ ĐH Stanford... (Vui lòng đợi vài giây)
Tải xong! Đang giải nén dữ liệu...
Giải nén hoàn tất!
Đang xây dựng đồ thị từ tập dữ liệu gốc...
--- THỐNG KÊ ĐỒ THỊ GỐC ---
Số lượng Node (Sản phẩm): 334,863
Số lượng Edge (Lượt mua chung): 925,872

Đang tìm kiếm 'Sản phẩm trung tâm' để làm điểm bắt đầu...
Sản phẩm trung tâm (Node ID: 548091) có 549 lượt mua kèm.
Đang trích xuất đồ thị con gồm 8,000 Nodes bằng thuật toán BFS...
Trích xuất Sub-graph thành công!

THỐNG KÊ NETWORK TOPOLOGY (Dành cho Chương 3 Báo Cáo)
1. Tổng số Nodes (Sản phẩm): 8,000
2. Tổng số Edges (Lượt mua chéo): 19,903
3. Density (Mật độ đồ thị): 0.00062
4. Đồ thị liên thông hoàn toàn (Connected).
Đang tính Average Shortest Path Length...
5. Average Shortest Path (Đường đi ngắn nhất trung bình): 6.2674

Đang lưu đồ thị vào file: ./amazon_graph.gpickle...
LƯU THÀNH CÔNG! ✅

[HƯỚNG DẪN BÀN GIAO]:
1. Thành hãy copy đoạn log ở CELL 5 và dán vào FILE BÁO CÁO - ĐẦY ĐỦ.
2. Ném fi